# Mean Reversion to VWAP on SPY
## Strategy Brief
This strategy aims to exploit mean reversion by comparing the current price of SPY to its Volume Weighted Average Price (VWAP). When the price is significantly below the VWAP, it may indicate a buying opportunity, while a price significantly above the VWAP may signal a selling opportunity. The strategy involves entering trades when the price deviates from the VWAP by a certain threshold and exiting when it reverts back. The results of this strategy will be evaluated against a buy-and-hold approach to determine its effectiveness.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters and constants that will be used throughout the strategy development and backtesting process. These include the start and end dates for data retrieval, the VWAP lookback period, and thresholds for trade signals.

In [ ]:
import datetime

# Constants
START_DATE = '2010-01-01'
END_DATE = datetime.datetime.today().strftime('%Y-%m-%d')
VWAP_LOOKBACK_PERIOD = 14  # in days
BUY_THRESHOLD = -0.02  # 2% below VWAP
SELL_THRESHOLD = 0.02  # 2% above VWAP

### PHASE 2 - Data Exploration
We will download historical price data for SPY from Yahoo Finance, calculate the VWAP, and visualize it alongside the price data to understand its behavior over time.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Calculate VWAP
data['Typical Price'] = (data['High'] + data['Low'] + data['Close']) / 3
data['VWAP'] = (data['Typical Price'] * data['Volume']).cumsum() / data['Volume'].cumsum()

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='Close Price')
plt.plot(data['VWAP'], label='VWAP', linestyle='--')
plt.title('SPY Price and VWAP')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
In this phase, we define the trading signals based on the deviation of the price from the VWAP. We will generate a signal series where a positive value indicates a buy signal and a negative value indicates a sell signal.

In [ ]:
# Generate signals
data['Signal'] = 0
price_to_vwap = (data['Close'] - data['VWAP']) / data['VWAP']
data.loc[price_to_vwap < BUY_THRESHOLD, 'Signal'] = 1  # Buy signal
data.loc[price_to_vwap > SELL_THRESHOLD, 'Signal'] = -1  # Sell signal

# Define positions
data['Position'] = data['Signal'].shift(1).fillna(0)

### PHASE 4 - Coding & Backtesting
We will calculate the daily returns based on the positions and plot the equity curve to visualize the performance of the strategy over time.

In [ ]:
# Calculate daily returns
data['Market Return'] = data['Close'].pct_change()
data['Strategy Return'] = data['Position'] * data['Market Return']
data['Equity Curve'] = (1 + data['Strategy Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity Curve'], label='Strategy Equity Curve')
plt.title('Equity Curve of Mean Reversion to VWAP Strategy')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
We will evaluate the performance of the strategy using key financial metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and Maximum Drawdown. We will compare these metrics against a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(equity_curve):
    # Calculate CAGR
    n_years = (equity_curve.index[-1] - equity_curve.index[0]).days / 365.25
    cagr = (equity_curve[-1] / equity_curve[0]) ** (1 / n_years) - 1
    
    # Calculate Sharpe Ratio
    sharpe_ratio = np.mean(data['Strategy Return']) / np.std(data['Strategy Return']) * np.sqrt(252)
    
    # Calculate Sortino Ratio
    downside_std = np.std(data['Strategy Return'][data['Strategy Return'] < 0])
    sortino_ratio = np.mean(data['Strategy Return']) / downside_std * np.sqrt(252)
    
    # Calculate Calmar Ratio
    max_drawdown = (equity_curve.cummax() - equity_curve).max()
    calmar_ratio = cagr / max_drawdown
    
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown = calculate_performance_metrics(data['Equity Curve'])

# Buy-and-hold metrics
buy_and_hold_cagr = (data['Close'][-1] / data['Close'][0]) ** (1 / n_years) - 1

# Print comparison table
comparison_table = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Mean Reversion': [cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown],
    'Buy and Hold': [buy_and_hold_cagr, '-', '-', '-', '-']
})
print(comparison_table)

### PHASE 6 - Deploy & Monitor
To deploy this strategy, we will create a function that downloads the last 60 days of SPY data, computes today's signal, and prints the recommended position.

In [ ]:
def get_today_signal():
    # Download last 60 days of SPY data
    recent_data = yf.download('SPY', start=(datetime.datetime.today() - datetime.timedelta(days=60)).strftime('%Y-%m-%d'), end=END_DATE)
    
    # Calculate VWAP
    recent_data['Typical Price'] = (recent_data['High'] + recent_data['Low'] + recent_data['Close']) / 3
    recent_data['VWAP'] = (recent_data['Typical Price'] * recent_data['Volume']).cumsum() / recent_data['Volume'].cumsum()
    
    # Calculate signal
    price_to_vwap = (recent_data['Close'] - recent_data['VWAP']) / recent_data['VWAP']
    signal = 0
    if price_to_vwap.iloc[-1] < BUY_THRESHOLD:
        signal = 1  # Buy
    elif price_to_vwap.iloc[-1] > SELL_THRESHOLD:
        signal = -1  # Sell
    
    print(f"Today's signal for SPY: {'Buy' if signal == 1 else 'Sell' if signal == -1 else 'Hold'}")

get_today_signal()